In [1]:
# IMPORT LIBRARIES
import pandas as pd 
import numpy as np 
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler 
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GridSearchCV,RandomizedSearchCV     # for hyperparamter tunning 
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix,accuracy_score,precision_score,recall_score,f1_score,classification_report

In [2]:
# data loading 
df=pd.read_csv(r"C:\Users\Admin\Downloads\credit_card_fraud_balanced.csv")
df.head()

,amount,transaction_hour,merchant_category,card_type,transaction_channel,location,distance_from_home_km,prev_transactions_24hr,card_age_months,customer_age,credit_score,account_status,is_fraud
0,14.44,11,Dining,Mastercard,Contactless,Ahmedabad,0.67,5,80,26,637,Active,0
1,185.75,12,Grocery,Mastercard,POS,Bengaluru,4.28,7,61,49,668,Active,0
2,245.32,7,Grocery,Amex,POS,Kolkata,7.74,4,86,38,628,New,0
3,304.05,9,Grocery,Mastercard,Online,Kolkata,10.00,3,62,30,676,Active,0
4,255.64,16,Healthcare,Mastercard,Online,Foreign,17.64,1,24,31,566,Active,0


In [4]:
# data cleaning 
df.isnull().sum()

amount                    0
transaction_hour          0
merchant_category         0
card_type                 0
transaction_channel       0
location                  0
distance_from_home_km     0
prev_transactions_24hr    0
card_age_months           0
customer_age              0
credit_score              0
account_status            0
is_fraud                  0
dtype: int64

In [5]:
# drop the duplicated values 
df.drop_duplicates(inplace=True)

In [6]:
df.head()

,amount,transaction_hour,merchant_category,card_type,transaction_channel,location,distance_from_home_km,prev_transactions_24hr,card_age_months,customer_age,credit_score,account_status,is_fraud
0,14.44,11,Dining,Mastercard,Contactless,Ahmedabad,0.67,5,80,26,637,Active,0
1,185.75,12,Grocery,Mastercard,POS,Bengaluru,4.28,7,61,49,668,Active,0
2,245.32,7,Grocery,Amex,POS,Kolkata,7.74,4,86,38,628,New,0
3,304.05,9,Grocery,Mastercard,Online,Kolkata,10.00,3,62,30,676,Active,0
4,255.64,16,Healthcare,Mastercard,Online,Foreign,17.64,1,24,31,566,Active,0


In [7]:
df.shape

(6230, 13)

In [8]:
# Value counts of categorcial columns
df["merchant_category"].value_counts(),df["card_type"].value_counts(),df["transaction_channel"].value_counts,df["location"].value_counts(),df["account_status"].value_counts()

(merchant_category
 Grocery          1150
 Dining            880
 Travel            695
 Electronics       678
 Online            668
 Entertainment     605
 Clothing          550
 Healthcare        450
 Fuel              254
 Utilities         157
 Jewelry           114
 Gaming             29
 Name: count, dtype: int64,
 card_type
 Visa          2332
 Mastercard    2115
 RuPay         1122
 Amex           661
 Name: count, dtype: int64,
 <bound method IndexOpsMixin.value_counts of 0       Contactless
 1               POS
 2               POS
 3            Online
 4            Online
            ...     
 6225    Contactless
 6226         Online
 6227    Contactless
 6228         Mobile
 6229         Mobile
 Name: transaction_channel, Length: 6230, dtype: object>,
 location
 Mumbai        1092
 Delhi          942
 Foreign        867
 Bengaluru      688
 Hyderabad      514
 Chennai        511
 Pune           483
 Kolkata        385
 Ahmedabad      329
 Jaipur         237
 Chandigarh    

In [9]:
# handling categorical columns 
label=LabelEncoder()
df["encoded_merchant_category"]=label.fit_transform(df["merchant_category"])
var=dict(zip(df["merchant_category"],df["encoded_merchant_category"]))
print(var)
df["card_type"]=df["card_type"].map({"Visa":0,"Mastercard":1,"RuPay":2,"Amex":3})
df["transaction_channel"]=df["transaction_channel"].map({"Online":0,"POS":1,"Contactless":2,"Mobile":3})
df["encoded_location"]=label.fit_transform(df["location"])
var1=dict(zip(df["location"],df["encoded_location"]))
print(var1)
df["account_status"]=df["account_status"].map({"Active":0,"New":1,"Premium":2,"Suspended":3,"Flagged":4})

{'Dining': 1, 'Grocery': 6, 'Healthcare': 7, 'Online': 9, 'Clothing': 0, 'Fuel': 4, 'Gaming': 5, 'Travel': 10, 'Electronics': 2, 'Utilities': 11, 'Jewelry': 8, 'Entertainment': 3}
{'Ahmedabad': 0, 'Bengaluru': 1, 'Kolkata': 8, 'Foreign': 5, 'Pune': 10, 'Chandigarh': 2, 'Chennai': 3, 'Mumbai': 9, 'Delhi': 4, 'Hyderabad': 6, 'Jaipur': 7}


In [10]:
df.head()

,amount,transaction_hour,merchant_category,card_type,transaction_channel,location,distance_from_home_km,prev_transactions_24hr,card_age_months,customer_age,credit_score,account_status,is_fraud,encoded_merchant_category,encoded_location
0,14.44,11,Dining,1,2.0,Ahmedabad,0.67,5,80,26,637,0,0,1,0
1,185.75,12,Grocery,1,1.0,Bengaluru,4.28,7,61,49,668,0,0,6,1
2,245.32,7,Grocery,3,1.0,Kolkata,7.74,4,86,38,628,1,0,6,8
3,304.05,9,Grocery,1,0.0,Kolkata,10.00,3,62,30,676,0,0,6,8
4,255.64,16,Healthcare,1,0.0,Foreign,17.64,1,24,31,566,0,0,7,5


In [11]:
# feature selection 
x=df.drop(columns=["is_fraud","merchant_category", "location"])
y=df["is_fraud"]

In [11]:
df.head()

,amount,transaction_hour,merchant_category,card_type,transaction_channel,location,distance_from_home_km,prev_transactions_24hr,card_age_months,customer_age,credit_score,account_status,is_fraud,encoded_merchant_category,encoded_location
0,14.44,11,Dining,1,2.0,Ahmedabad,0.67,5,80,26,637,0,0,1,0
1,185.75,12,Grocery,1,1.0,Bengaluru,4.28,7,61,49,668,0,0,6,1
2,245.32,7,Grocery,3,1.0,Kolkata,7.74,4,86,38,628,1,0,6,8
3,304.05,9,Grocery,1,0.0,Kolkata,10.00,3,62,30,676,0,0,6,8
4,255.64,16,Healthcare,1,0.0,Foreign,17.64,1,24,31,566,0,0,7,5


In [12]:
# data scaling 
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [13]:
# data scaling 
scaler=StandardScaler()
x_test_scale=scaler.fit_transform(x_test)
x_train_scale=scaler.transform(x_train)

In [14]:
x_test_scale

array([[-0.28328918, -0.29961743,  0.94621653, ...,  0.30228985,
         0.57031613,  1.20088289],
       [-0.08615136,  0.74221509, -0.04286551, ..., -0.65368261,
        -0.57583532, -1.44685005],
       [-0.28431399,  0.56857634,  0.94621653, ..., -0.65368261,
        -1.14891104, -0.78491682],
       ...,
       [-0.18803701, -1.34144996, -0.04286551, ..., -0.65368261,
         0.28377827, -0.78491682],
       [-0.23637014, -0.47325619, -0.04286551, ..., -0.65368261,
        -0.86237318, -0.78491682],
       [-0.27621875, -0.99417245,  0.94621653, ..., -0.65368261,
         0.57031613,  1.53184951]], shape=(1246, 12))

In [15]:
x_train_scale

array([[-0.02958794,  0.74221509, -0.04286551, ..., -0.65368261,
         1.14339185, -1.44685005],
       [-0.27327141, -0.82053369, -0.04286551, ...,  0.30228985,
         0.28377827, -1.11588344],
       [-0.25271156, -0.82053369, -0.04286551, ..., -0.65368261,
         0.57031613,  1.20088289],
       ...,
       [-0.2598376 ,  1.26313135,  1.93529856, ..., -0.65368261,
        -0.57583532,  1.20088289],
       [-0.07676119, -0.64689494, -0.04286551, ..., -0.65368261,
        -0.86237318,  1.53184951],
       [-0.28765854,  0.91585385,  0.94621653, ..., -0.65368261,
        -1.14891104,  0.20798304]], shape=(4984, 12))

In [16]:
# paramter for random forest tunning 
para_random={"max_depth":[4,6,8],"n_estimators":[100,150,200]}

In [17]:
tunning_model=RandomForestClassifier(random_state=42)

In [18]:
# model using Random forest
# model selction (Initilaize)
grid_search=GridSearchCV(estimator=tunning_model,
                         param_grid=para_random,
                        cv=5,
                        n_jobs=-1,
                        scoring="accuracy",
                        verbose=1)    # for classfication scoring will be accuracy_score
# model traning 
grid_search.fit(x_train_scale,y_train)

Fitting 5 folds for each of 9 candidates, totalling 45 fits


,estimator,RandomForestC...ndom_state=42)
,param_grid,"{'max_depth': [4, 6, ...], 'n_estimators': [100, 150, ...]}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,5
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,150


In [19]:
# check the best combination 
grid_search.best_params_

{'max_depth': 8, 'n_estimators': 150}

In [20]:
model=grid_search.best_estimator_

In [21]:
# predict 
y_pred=model.predict(x_test_scale)
accuracy_score(y_test,y_pred)

0.9951845906902087

In [22]:
# traning accuracy 
y_pred_train=model.predict(x_train_scale)
accuracy_score(y_train,y_pred_train)

0.997792937399679

# PREDECTIVE MODEL 


In [24]:
# amount	transaction_hour	transaction_hour	card_type	transaction_channel	location	distance_from_home_km	prev_transactions_24hr	card_age_months	customer_age	credit_score	account_status	encoded_merchant_category	encoded_locatioN
amount=int(input("Enter Amount : "))
hour=int(input("Enter Transaction hour : "))
card=int(input("Enter card type ---> Visa:0,Mastercard:1,RuPay:2,Amex:3 : "))
merchant=int(input("Enter Merchant category ---->  Dining: 1, 'Grocery': 6, 'Healthcare': 7, 'Online': 9, 'Clothing': 0, 'Fuel': 4,'Gaming': 5, 'Travel': 10, 'Electronics': 2, 'Utilities': 11, 'Jewelry': 8, 'Entertainment': 3"))
channel=int(input("Enter Transaction Channel ---> Online:0,POS:1,Contactless:2,ATM:3,Mobile:4} "))
location=int(input("Enter Location --->'Ahmedabad': 0, 'Bengaluru': 1, 'Kolkata': 8, 'Foreign': 5, 'Pune': 10, 'Chandigarh': 2, 'Chennai': 3, 'Mumbai': 9, 'Delhi': 4, 'Hyderabad': 6, 'Jaipur': 7  : "))
distance=int(input("Enter distance from home :"))
prev=int(input("Enter prev_transactions_24hr : "))
age=int(input("Enter card age month"))
cust_age=int(input("Enter Customer age : "))
score=int(input("Enter credit_score :"))
status=int(input("Enter Account status ---->  Active:0, New:1, Premium:2, Suspended:3, Flagged:4  "))
data=scaler.transform([[amount,hour,card,merchant,channel,location,distance,prev,age,cust_age,score,status]])
pred=model.predict(data)
print("Fraud : ",pred)

Enter Amount :  5000
Enter Transaction hour :  14
Enter card type ---> Visa:0,Mastercard:1,RuPay:2,Amex:3 :  0
Enter Merchant category ---->  Dining: 1, 'Grocery': 6, 'Healthcare': 7, 'Online': 9, 'Clothing': 0, 'Fuel': 4,'Gaming': 5, 'Travel': 10, 'Electronics': 2, 'Utilities': 11, 'Jewelry': 8, 'Entertainment': 3 6
Enter Transaction Channel ---> Online:0,POS:1,Contactless:2,ATM:3,Mobile:4}  0
Enter Location --->'Ahmedabad': 0, 'Bengaluru': 1, 'Kolkata': 8, 'Foreign': 5, 'Pune': 10, 'Chandigarh': 2, 'Chennai': 3, 'Mumbai': 9, 'Delhi': 4, 'Hyderabad': 6, 'Jaipur': 7  :  0
Enter distance from home : 5
Enter prev_transactions_24hr :  2
Enter card age month 12
Enter Customer age :  30
Enter credit_score : 750
Enter Account status ---->  Active:0, New:1, Premium:2, Suspended:3, Flagged:4   0


Fraud :  [1]


C:\Users\Admin\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
